# AdTech Clickstream Analytics Dataset

Since real-world advertising datasets from companies are not publicly available due to privacy and business restrictions, this dataset was synthetically generated to simulate a realistic AdTech environment.

The goal of creating this dataset is to replicate how advertising platforms track user interactions from clicks to installs and revenue events. The dataset is designed with relational tables to reflect real production data structures commonly used in advertising analytics.

Additionally, several real-world data challenges were intentionally introduced, including:
- Fraudulent traffic patterns (bot clicks, click bursts, suspicious publishers)
- Missing values
- Formatting inconsistencies
- Duplicate records
- Outlier values

These characteristics make the dataset suitable for practicing real-world data analytics tasks such as data cleaning, fraud detection, campaign performance analysis, and user conversion analysis.

The dataset consists of three core tables:
- **Clicks Table** – contains click-level advertising traffic data
- **Installs Table** – records app installs linked to clicks
- **Revenue Table** – tracks post-install revenue events

This structure enables relational analysis using SQL joins and simulates a typical analytics workflow used in advertising technology platforms.

In [ ]:
# install dependencies
!pip install faker tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 14.7 MB/s eta 0:00:00


In [ ]:
# install dependencies
!pip install pandas numpy tqdm pyarrow

In [ ]:
# import libraries
import pandas as pd
import numpy as np
from faker import Faker
from tqdm import tqdm
import random

fake = Faker()

In [ ]:
# marketing campaign data generation
n_clicks = 5000000  

device_types = ['Android','iOS','Tablet']
countries = ['US','India','UK','Canada','Germany']

campaigns = list(range(1,201))
publishers = list(range(1,101))
users = list(range(1,1000001))

In [ ]:
# function to generate random IP address
def random_ip():
    return ".".join(str(random.randint(0,255)) for _ in range(4))

In [ ]:
# generate synthetic click data
clicks = pd.DataFrame({
    'click_id': np.arange(1, n_clicks+1),
    'campaign_id': np.random.choice(campaigns,n_clicks),
    'publisher_id': np.random.choice(publishers,n_clicks),
    'user_id': np.random.choice(users,n_clicks),
    'device_type': np.random.choice(device_types,n_clicks,p=[0.6,0.3,0.1]),
    'country': np.random.choice(countries,n_clicks),
    'cost': np.round(np.random.uniform(0.01,0.5,n_clicks),2)})

clicks['timestamp'] = pd.to_datetime(
    np.random.randint(
        pd.Timestamp('2024-01-01').value//10**9,
        pd.Timestamp('2024-12-31').value//10**9,
        n_clicks),unit='s')

clicks['ip_address'] = [random_ip() for _ in range(n_clicks)]

clicks.head()

,click_id,campaign_id,publisher_id,user_id,device_type,country,cost,timestamp,ip_address
0,1,200,25,871813,Android,India,0.31,2024-07-16 13:37:23,161.169.61.254
1,2,71,49,154022,iOS,Germany,0.09,2024-12-21 19:18:45,180.74.181.188
2,3,22,42,329945,iOS,UK,0.32,2024-05-09 08:18:11,219.249.164.50
3,4,48,2,893142,Android,India,0.21,2024-05-11 17:54:53,131.119.190.132
4,5,143,6,998002,iOS,UK,0.19,2024-05-18 01:00:44,114.157.194.141


In [ ]:
# generate install data
install_rate = 0.05

install_clicks = clicks.sample(frac=install_rate)

installs = pd.DataFrame({
    'install_id': np.arange(1,len(install_clicks)+1),
    'click_id': install_clicks['click_id']})

installs['install_timestamp'] = install_clicks['timestamp'] + pd.to_timedelta(
    np.random.randint(10,3600,len(installs)),unit='s')

installs.head()

,install_id,click_id,install_timestamp
998346,1,998347,2024-10-14 04:28:26
71934,2,71935,2024-05-03 14:45:17
2877631,3,2877632,2024-02-04 16:59:07
1310017,4,1310018,2024-09-06 18:41:15
4586025,5,4586026,2024-08-13 03:01:15


In [ ]:
# generate revenue data
paying_installs = installs.sample(frac=0.4)

revenue = pd.DataFrame({'user_id': clicks.loc[
        clicks['click_id'].isin(paying_installs['click_id']),'user_id'].values,
    'revenue': np.round(np.random.exponential(20,len(paying_installs)),2)})

revenue['event_timestamp'] = pd.to_datetime(
    np.random.randint(
        pd.Timestamp('2024-01-01').value//10**9,
        pd.Timestamp('2024-12-31').value//10**9,
        len(revenue)),unit="s")

revenue['event_type'] = np.random.choice(
    ['purchase','subscription','ad_view'],
    len(revenue)
)

revenue.head()

,user_id,revenue,event_timestamp,event_type
0,828722,1.85,2024-02-12 10:26:31,subscription
1,460519,34.02,2024-08-23 03:20:06,ad_view
2,316533,21.28,2024-04-21 12:45:48,ad_view
3,304636,2.45,2024-05-09 09:11:27,subscription
4,39495,4.39,2024-04-18 04:58:57,ad_view


In [ ]:
# save datasets to parquet
clicks.to_parquet('clicks.parquet',index=False)
installs.to_parquet('installs.parquet',index=False)
revenue.to_parquet('revenue.parquet',index=False)

In [ ]:
#check
df1 = pd.read_parquet('installs.parquet')

In [ ]:
df1.head()
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250000 entries, 0 to 249999
Data columns (total 3 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   install_id         250000 non-null  int64         
 1   click_id           250000 non-null  int64         
 2   install_timestamp  250000 non-null  datetime64[ns]
dtypes: datetime64[ns](1), int64(2)
memory usage: 5.7 MB


In [ ]:
# check shapes
print(pd.read_parquet('clicks.parquet').shape)
print(pd.read_parquet('installs.parquet').shape)
print(pd.read_parquet('revenue.parquet').shape)

(5000000, 9)
(250000, 3)


(100000, 4)

In [ ]:
#lets add some bot traffic to make it more realistic
bot_ips = clicks['ip_address'].sample(40).tolist()
bot_rows = clicks.sample(60000)
bot_rows['ip_address'] = np.random.choice(bot_ips,len(bot_rows))

clicks.update(bot_rows)

In [ ]:
# add some fraudulent publishers
fraud_publishers = np.random.choice(clicks['publisher_id'].unique(),6)
mask = clicks['publisher_id'].isin(fraud_publishers)
clicks.loc[mask,'cost'] = np.random.uniform(0.4,0.7)

In [ ]:
# add some click bursts
burst_rows = clicks.sample(40000)

clicks.loc[burst_rows.index,'timestamp'] = pd.Timestamp("2024-06-01 12:00:00")

In [ ]:
# add some noise to install timestamps
installs['install_timestamp'] = installs['install_timestamp'] - pd.to_timedelta(
    np.random.randint(1,5,len(installs)),unit="s")

In [ ]:
# add some missing values
mask = np.random.rand(len(clicks)) < 0.02
clicks.loc[mask,'country'] = None

mask2 = np.random.rand(len(clicks)) < 0.01
clicks.loc[mask2,'device_type'] = None

In [ ]:
# add some outliers
rows = clicks.sample(8000).index
clicks.loc[rows,'device_type'] = "android"

rows2 = clicks.sample(8000).index
clicks.loc[rows2,'country'] = "usa"

In [ ]:
# add some duplicates
duplicates = clicks.sample(5000)

clicks = pd.concat([clicks,duplicates],ignore_index=True)

In [ ]:
# add some outliers in cost
rows = clicks.sample(3000).index

clicks.loc[rows,'cost'] = np.random.uniform(5,20)

In [ ]:
# save the messy datasets
clicks.to_parquet('clicks_messy.parquet',index=False)
installs.to_parquet('installs_messy.parquet',index=False)
revenue.to_parquet('revenue_messy.parquet',index=False)

In [ ]:
#check
clicks.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5005000 entries, 0 to 5004999
Data columns (total 9 columns):
 #   Column        Dtype         
---  ------        -----         
 0   click_id      int64         
 1   campaign_id   int64         
 2   publisher_id  int64         
 3   user_id       int64         
 4   device_type   object        
 5   country       object        
 6   cost          float64       
 7   timestamp     datetime64[ns]
 8   ip_address    object        
dtypes: datetime64[ns](1), float64(1), int64(4), object(3)
memory usage: 343.7+ MB


In [ ]:
#load data
df3 = pd.read_parquet('clicks_messy.parquet')
df4 = pd.read_parquet('installs_messy.parquet')
df5 = pd.read_parquet('revenue_messy.parquet')

In [ ]:
#check
print(df3.info())
print(df4.info())
print(df5.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5005000 entries, 0 to 5004999
Data columns (total 9 columns):
 #   Column        Dtype         
---  ------        -----         
 0   click_id      int64         
 1   campaign_id   int64         
 2   publisher_id  int64         
 3   user_id       int64         
 4   device_type   object        
 5   country       object        
 6   cost          float64       
 7   timestamp     datetime64[ns]
 8   ip_address    object        
dtypes: datetime64[ns](1), float64(1), int64(4), object(3)
memory usage: 343.7+ MB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250000 entries, 0 to 249999
Data columns (total 3 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   install_id         250000 non-null  int64         
 1   click_id           250000 non-null  int64         
 2   install_timestamp  250000 non-null  datetime64[ns]
dtypes: datetime64[ns](1), int64(2)


In [ ]:
#check duplicates
print(df3.duplicated().sum())
print(df4.duplicated().sum())
print(df5.duplicated().sum())

4996
0
0


In [ ]:
#shape check
print(df3.shape)
print(df4.shape)
print(df5.shape)

(5005000, 9)
(250000, 3)
(100000, 4)
